# PyTorch Autograd Basics

In this notebook, We will learn:

- what autograd is
- why gradients are important in deep learning
- how PyTorch computes gradients automatically
- how to use `requires_grad`
- how `.backward()` works
- how to access gradients using `.grad`
- why gradients accumulate
- how to stop gradient tracking

---

## 1. What is Autograd?

Autograd is PyTorch’s automatic differentiation system.

It automatically calculates gradients for tensors that have:

```python
requires_grad=True
```

These gradients are useful because deep learning models learn by updating weights using gradients.

In simple words:

forward pass → calculate output

backward pass → calculate gradients

gradients → tell us how to update parameters



---


## 2. Why is Autograd Important?

In machine learning and deep learning, we need gradients to minimize loss.

Without autograd, we would need to calculate derivatives manually for every operation.

That becomes difficult when the model is large.

PyTorch autograd makes this easy:
- it tracks operations
- builds a computation graph
- computes gradients automatically using backpropagation

---

## 3. A Simple Mathematical Example

Before using PyTorch, let us start with a simple function:

$$
y = x^2
$$

Its derivative is:

$$
\frac{dy}{dx} = 2x
$$

This helps us compare manual differentiation with PyTorch autograd.

In Code :
```python
def dy_dx(x):
    return 2 * x

print("Manual derivative at x = 3:", dy_dx(3))
```

---

## 4. First Autograd Example

Now we will let PyTorch compute the derivative of:

$$
y = x^2
$$

at $ x = 3 $

In [1]:
import torch

x = torch.tensor(3.0, requires_grad=True)
y = x ** 2

print("x:", x)
print("y:", y)

x: tensor(3., requires_grad=True)
y: tensor(9., grad_fn=<PowBackward0>)


In [2]:
y.backward()
print("Gradient dy/dx at x = 3:", x.grad)

Gradient dy/dx at x = 3: tensor(6.)


### Reason

- `requires_grad=True` tells PyTorch to track operations on `x`
- `y = x**2` creates a computation graph
- `y.backward()` computes the gradient
- `x.grad` stores the derivative of `y` with respect to `x`

Since:

$$
\frac{dy}{dx} = 2x
$$

at $x=3$, the gradient should be 6.

## 5. Autograd with Chain Rule

Now let us take a slightly more complex example:

$$
y = x^2
$$
$$
z = \sin(y)
$$

So:

$$
z = \sin(x^2)
$$

Using chain rule:

$$
\frac{dz}{dx} = \cos(x^2) \cdot 2x
$$

We will compare manual calculation with autograd.

In [3]:
import math

def dz_dx(x):
    return 2 * x * math.cos(x ** 2)

print("Manual derivative at x = 4:", dz_dx(4))

Manual derivative at x = 4: -7.661275842587077


In [4]:
x = torch.tensor(4.0, requires_grad=True)
y = x ** 2
z = torch.sin(y)

print("x:", x)
print("y:", y)
print("z:", z)

x: tensor(4., requires_grad=True)
y: tensor(16., grad_fn=<PowBackward0>)
z: tensor(-0.2879, grad_fn=<SinBackward0>)


In [5]:
z.backward()
print("Autograd gradient dz/dx at x = 4:", x.grad)

Autograd gradient dz/dx at x = 4: tensor(-7.6613)


### Reason

This example shows the real strength of autograd.

Even when multiple operations are connected, PyTorch automatically applies the chain rule and computes the correct gradient.

## 6. Why is `y.grad` Usually None?

In PyTorch, gradients are normally stored only for **leaf tensors**.

A leaf tensor is usually a tensor created directly by the user with:

```python
requires_grad=True
```

Example:

* x is a leaf tensor

* y = x**2 is not a leaf tensor

So after backward():

* x.grad is available

* y.grad is usually None


### Code cell
```python
x = torch.tensor(4.0, requires_grad=True)
y = x ** 2
z = torch.sin(y)

z.backward()

print("x.grad:", x.grad)
print("y.grad:", y.grad)
```

In [6]:
x = torch.tensor(4.0, requires_grad=True)
y = x ** 2
z = torch.sin(y)

z.backward()

print("x.grad:", x.grad)
print("y.grad:", y.grad)

x.grad: tensor(-7.6613)
y.grad: None


/tmp/ipykernel_708/4217855221.py:8: UserWarning: The .grad attribute of a Tensor that is not a leaf Tensor is being accessed. Its .grad attribute won't be populated during autograd.backward(). If you indeed want the .grad field to be populated for a non-leaf Tensor, use .retain_grad() on the non-leaf Tensor. If you access the non-leaf Tensor by mistake, make sure you access the leaf Tensor instead. See github.com/pytorch/pytorch/pull/30531 for more information. (Triggered internally at /pytorch/build/aten/src/ATen/core/TensorBody.h:492.)
  print("y.grad:", y.grad)


## 7. Manual Gradient vs Autograd

Now let us use a small logistic-style example.

We will:
- do a forward pass
- calculate binary cross-entropy loss
- compute gradients manually
- compare them with PyTorch autograd

This helps us understand why autograd is helpful.

In [7]:
import torch

# input and target
x = torch.tensor(6.7)
y = torch.tensor(0.0)

# parameters
w = torch.tensor(1.0)
b = torch.tensor(0.0)

def binary_cross_entropy_loss(prediction, target):
    epsilon = 1e-8
    prediction = torch.clamp(prediction, epsilon, 1 - epsilon)
    return -(target * torch.log(prediction) + (1 - target) * torch.log(1 - prediction))


# forward pass
z = w * x + b
y_pred = torch.sigmoid(z)
loss = binary_cross_entropy_loss(y_pred, y)

print("Prediction:", y_pred.item())
print("Loss:", loss.item())

Prediction: 0.998770534992218
Loss: 6.701176166534424


In [8]:
# manual gradients
dloss_dy_pred = (y_pred - y) / (y_pred * (1 - y_pred))
dy_pred_dz = y_pred * (1 - y_pred)
dz_dw = x
dz_db = 1

dL_dw = dloss_dy_pred * dy_pred_dz * dz_dw
dL_db = dloss_dy_pred * dy_pred_dz * dz_db

print("Manual gradient w.r.t weight:", dL_dw.item())
print("Manual gradient w.r.t bias:", dL_db.item())

Manual gradient w.r.t weight: 6.691762447357178
Manual gradient w.r.t bias: 0.998770534992218


In [9]:
# autograd version
x = torch.tensor(6.7)
y = torch.tensor(0.0)

w = torch.tensor(1.0, requires_grad=True)
b = torch.tensor(0.0, requires_grad=True)

z = w * x + b
y_pred = torch.sigmoid(z)
loss = binary_cross_entropy_loss(y_pred, y)

loss.backward()

print("Autograd gradient w.r.t weight:", w.grad.item())
print("Autograd gradient w.r.t bias:", b.grad.item())

Autograd gradient w.r.t weight: 6.6917619705200195
Autograd gradient w.r.t bias: 0.9987704753875732


### Reason

This section proves that PyTorch autograd gives the same gradients that we get manually.

For small examples, manual differentiation is possible.

For large neural networks, manual gradients become difficult, so autograd becomes essential.

## 8. Gradients for Multiple Values

Autograd also works for tensors, not only single numbers.

Here we will compute the gradient of:

$$
y = \text{mean}(x^2)
$$

where `x` is a vector.

In [10]:
x = torch.tensor([1.0, 2.0, 3.0], requires_grad=True)
y = (x ** 2).mean()

print("x:", x)
print("y:", y)

x: tensor([1., 2., 3.], requires_grad=True)
y: tensor(4.6667, grad_fn=<MeanBackward0>)


In [11]:
y.backward()
print("Gradient of x:", x.grad)

Gradient of x: tensor([0.6667, 1.3333, 2.0000])


### Reason

This shows that PyTorch computes gradients element by element.

Since:

$$
y = \frac{x_1^2 + x_2^2 + x_3^2}{3}
$$

the gradient becomes:

$$
\frac{2x}{3}
$$

So autograd works naturally even for vectors.

## 9. Gradients Accumulate in PyTorch

PyTorch adds new gradients to the old gradients.

That means if we call `.backward()` multiple times without clearing gradients, the gradient values accumulate.

In [12]:
x = torch.tensor(2.0, requires_grad=True)

y = x ** 2
y.backward()
print("After first backward:", x.grad)

y = x ** 2
y.backward()
print("After second backward:", x.grad)

After first backward: tensor(4.)
After second backward: tensor(8.)


### Reason

After the first backward pass, gradient is 4.

After the second backward pass, PyTorch adds the new gradient again, so it becomes 8.

That is why gradients usually need to be cleared before the next backward pass during training.

## 10. Clearing Gradients

To prevent gradient accumulation, we clear gradients before the next backward pass.

In [13]:
x = torch.tensor(2.0, requires_grad=True)

y = x ** 2
y.backward()
print("Gradient before zeroing:", x.grad)

x.grad.zero_()
print("Gradient after zeroing:", x.grad)

Gradient before zeroing: tensor(4.)
Gradient after zeroing: tensor(0.)


### Reason

In model training, gradients are computed again and again in every iteration.

So clearing old gradients is necessary before computing new ones.

## 11. How to Stop Gradient Tracking

Sometimes we do not want PyTorch to track gradients.

This is useful during:
- inference
- testing
- saving memory
- faster execution

Common ways:
- `requires_grad_(False)`
- `detach()`
- `torch.no_grad()`

### A. requires_grad_(False)

This turns off gradient tracking for that tensor.

In [14]:
x = torch.tensor(2.0, requires_grad=True)
x.requires_grad_(False)

y = x ** 2
print("y:", y)
print("Does y require grad?", y.requires_grad)

y: tensor(4.)
Does y require grad? False


### B. detach()

`detach()` creates a new tensor that is disconnected from the computation graph.

In [15]:
x = torch.tensor(2.0, requires_grad=True)
z = x.detach()

y = x ** 2
y1 = z ** 2

print("y requires grad:", y.requires_grad)
print("y1 requires grad:", y1.requires_grad)

y requires grad: True
y1 requires grad: False


### C. torch.no_grad()

`torch.no_grad()` is commonly used during prediction or evaluation when gradients are not needed.

In [16]:
x = torch.tensor(2.0, requires_grad=True)

with torch.no_grad():
    y = x ** 2

print("y:", y)
print("Does y require grad?", y.requires_grad)

y: tensor(4.)
Does y require grad? False


## 12. Important Caution

If a tensor does not require gradients, then calling:

```python
tensor.backward()
```


---

## 13. Summary

In this notebook, I learned:

- Autograd is PyTorch’s automatic differentiation system
- `requires_grad=True` tells PyTorch to track operations
- `.backward()` computes gradients
- `.grad` stores gradients for leaf tensors
- PyTorch automatically applies the chain rule
- gradients accumulate unless we clear them
- gradients can be cleared using `.zero_()`
- gradient tracking can be stopped using `requires_grad_(False)`, `detach()`, or `torch.no_grad()`